# 08 — Análisis y Visualización Gold

**Proyecto:** Mexico Economic Pipeline  
**Propósito:** Leer las tablas Gold y generar análisis con tendencias,
cruces de fuentes y visualizaciones para el reporte final.

**Pipeline:** `...` → `07_capa_oro.ipynb` → **`08_analisis_gold.ipynb`**

**Prerequisito:** haber ejecutado `07_capa_oro.ipynb` completo.

---

## Preguntas analíticas

Este notebook responde tres preguntas centrales sobre el mercado laboral mexicano:

**P1 — ¿Cómo ha evolucionado el poder adquisitivo de los trabajadores mexicanos entre 2005 y 2026?**  
Comparamos el salario real (deflactado con INPC base T4-2018) contra la inflación anual.
Los cálculos de deflación se realizan una sola vez en el pipeline ETL (`07_capa_oro.ipynb`);
aquí solo consultamos los resultados.

**P2 — ¿Ha disminuido la brecha salarial de género en las últimas dos décadas?**  
Medimos la diferencia porcentual entre el salario real promedio de hombres y mujeres por año,
desagregando por grupo de edad para detectar si el cierre (o ampliación) es uniforme.

**P3 — ¿Existe una relación entre el crecimiento económico (IGAE) y la mejora del bienestar laboral?**  
Correlacionamos el índice de actividad económica con el salario real y la tasa de desocupación,
buscando si el crecimiento macroeconómico se traduce en mejoras para los trabajadores.

---

**Análisis generados:**
```
1. Evolución del salario real 2005–2026          → P1
2. Brecha salarial de género                     → P2
3. Inflación vs poder adquisitivo                → P1
4. Empleo por sector económico                   → P3
5. Tasa de desocupación — ciclos económicos      → P3
6. IGAE vs salario real — crecimiento y bienestar → P3
7. Salario real por nivel educativo              → P1 / P2
8. Resumen estadístico y hallazgos               → todas
```

## 0 · Imports y configuración

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('08_analisis_gold')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

# ── Estilo global de gráficas ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
})
COLORES = ['#2563eb','#dc2626','#16a34a','#d97706','#7c3aed',
           '#0891b2','#be185d','#65a30d','#b45309','#4338ca']

print('Spark', spark.version)

In [ ]:
# ── Rutas Gold ───────────────────────────────────────────────────────────────
BASE  = Path(os.getcwd()).parent
GOLD  = BASE / 'data' / 'gold'

GOLD_SALARIO = str(GOLD / 'salario_real')
GOLD_EMPLEO  = str(GOLD / 'empleo_sector')
GOLD_MACRO   = str(GOLD / 'macro_trimestral')
GOLD_BRECHA  = str(GOLD / 'brecha_salarial')
GOLD_KPI     = str(GOLD / 'kpi_anual')

FIGS = BASE / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

print('Gold:', str(GOLD))
print('Figs:', str(FIGS))

---
## 1 · Cargar tablas Gold

In [ ]:
# ── 1.1 Leer parquets ────────────────────────────────────────────────────────
df_salario = spark.read.parquet(GOLD_SALARIO)
df_empleo  = spark.read.parquet(GOLD_EMPLEO)
df_macro   = spark.read.parquet(GOLD_MACRO)
df_brecha  = spark.read.parquet(GOLD_BRECHA)
df_kpi     = spark.read.parquet(GOLD_KPI)

for nombre, df in [('salario', df_salario), ('empleo', df_empleo),
                   ('macro',   df_macro),   ('brecha', df_brecha),
                   ('kpi',     df_kpi)]:
    print(f'{nombre:<10} {df.count():>6,} filas  {len(df.columns)} cols')

In [ ]:
# ── 1.2 Convertir a Pandas para visualización ────────────────────────────────
pd_salario = df_salario.toPandas()
pd_empleo  = df_empleo.toPandas()
pd_macro   = df_macro.toPandas()
pd_brecha  = df_brecha.toPandas()
pd_kpi     = df_kpi.toPandas()

# KPI anual ordenado
pd_kpi = pd_kpi.sort_values('anio').reset_index(drop=True)
pd_macro = pd_macro.sort_values(['anio','trimestre']).reset_index(drop=True)
pd_macro['periodo'] = pd_macro['anio'].astype(str) + '-T' + pd_macro['trimestre'].astype(str)

print('Pandas OK — rango:', pd_kpi['anio'].min(), '–', pd_kpi['anio'].max())

---
## 2 · Evolución del salario real 2005–2026

Promedio anual del salario real (pesos constantes base T4-2018).

In [ ]:
# ── Preparar: salario promedio anual por sexo ────────────────────────────────
sal_anual = (
    pd_salario
    .groupby(['anio', 'sexo'])['salario_real_prom']
    .mean()
    .reset_index()
)
sal_total = (
    pd_salario
    .groupby('anio')['salario_real_prom']
    .mean()
    .reset_index()
    .rename(columns={'salario_real_prom': 'total'})
)

hombres = sal_anual[sal_anual['sexo'] == 'Hombre']
mujeres = sal_anual[sal_anual['sexo'] == 'Mujer']

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(sal_total['anio'], sal_total['total'],
        color='#1e3a5f', linewidth=2.5, label='Total', zorder=3)
ax.plot(hombres['anio'], hombres['salario_real_prom'],
        color=COLORES[0], linewidth=1.8, linestyle='--', label='Hombre')
ax.plot(mujeres['anio'], mujeres['salario_real_prom'],
        color=COLORES[1], linewidth=1.8, linestyle='--', label='Mujer')

# Marcar eventos clave
for anio, label, color in [
    (2009, 'Crisis\n2008–09', '#f59e0b'),
    (2017, 'Reforma\nlaboral', '#6366f1'),
    (2020, 'COVID-19', '#ef4444'),
    (2023, 'Aumento\nSMG', '#16a34a'),
]:
    ax.axvline(anio, color=color, linewidth=1.2, linestyle=':', alpha=0.8)
    ax.text(anio + 0.1, ax.get_ylim()[0] * 1.02 if ax.get_ylim()[0] > 0 else 100,
            label, fontsize=8, color=color, va='bottom')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Año')
ax.set_ylabel('Salario real promedio (pesos base T4-2018)')
ax.set_title('Evolución del Salario Real en México 2005–2026', fontweight='bold', pad=12)
ax.legend(framealpha=0.9)
ax.set_xlim(2005, 2026)

plt.tight_layout()
plt.savefig(FIGS / '01_salario_real_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 01_salario_real_evolucion.png')

---
## 3 · Brecha salarial de género 2005–2026

In [ ]:
# ── Brecha promedio anual ────────────────────────────────────────────────────
brecha_anual = (
    pd_brecha
    .groupby('anio')[['sal_real_hombre', 'sal_real_mujer', 'brecha_pct']]
    .mean()
    .reset_index()
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Panel superior: salario H vs M
ax1.fill_between(brecha_anual['anio'],
                 brecha_anual['sal_real_mujer'],
                 brecha_anual['sal_real_hombre'],
                 alpha=0.15, color=COLORES[0], label='Brecha')
ax1.plot(brecha_anual['anio'], brecha_anual['sal_real_hombre'],
         color=COLORES[0], linewidth=2, label='Hombre')
ax1.plot(brecha_anual['anio'], brecha_anual['sal_real_mujer'],
         color=COLORES[1], linewidth=2, label='Mujer')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.set_ylabel('Salario real (pesos base T4-2018)')
ax1.set_title('Brecha Salarial de Género en México 2005–2026', fontweight='bold', pad=12)
ax1.legend(framealpha=0.9)

# Panel inferior: brecha %
ax2.bar(brecha_anual['anio'], brecha_anual['brecha_pct'],
        color=COLORES[0], alpha=0.75, width=0.7)
ax2.axhline(brecha_anual['brecha_pct'].mean(), color='red',
            linestyle='--', linewidth=1.2,
            label=f'Promedio {brecha_anual["brecha_pct"].mean():.1f}%')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax2.set_ylabel('Brecha (% sobre salario masculino)')
ax2.set_xlabel('Año')
ax2.legend(framealpha=0.9)
ax2.set_xlim(2004.5, 2026.5)

plt.tight_layout()
plt.savefig(FIGS / '02_brecha_salarial_genero.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 02_brecha_salarial_genero.png')

---
## 4 · Inflación vs salario real — poder adquisitivo

In [ ]:
# ── Cruce INPC + salario real (KPI anual) ───────────────────────────────────
kpi = pd_kpi.dropna(subset=['inflacion_anual_pct', 'var_salario_real_pct'])

fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(kpi))
w = 0.35

bars1 = ax.bar(x - w/2, kpi['inflacion_anual_pct'],
               width=w, label='Inflación anual (%)', color=COLORES[1], alpha=0.8)
bars2 = ax.bar(x + w/2, kpi['var_salario_real_pct'],
               width=w, label='Var. salario real (%)',
               color=[COLORES[2] if v >= 0 else COLORES[1]
                      for v in kpi['var_salario_real_pct']], alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(kpi['anio'].astype(int), rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_ylabel('Variación anual (%)')
ax.set_title('Inflación vs Variación del Salario Real 2006–2026',
             fontweight='bold', pad=12)
ax.legend(framealpha=0.9)

# Anotar años con salario real negativo (pérdida de poder adquisitivo)
for i, row in kpi.reset_index(drop=True).iterrows():
    if row['var_salario_real_pct'] < -2:
        ax.annotate(f"{row['var_salario_real_pct']:.1f}%",
                    xy=(i + w/2, row['var_salario_real_pct'] - 0.3),
                    ha='center', va='top', fontsize=8, color='darkred')

plt.tight_layout()
plt.savefig(FIGS / '03_inflacion_vs_salario.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 03_inflacion_vs_salario.png')

---
## 5 · Empleo por sector económico

In [ ]:
# ── Top sectores por ocupados (promedio anual) ───────────────────────────────
sector_anual = (
    pd_empleo
    .groupby(['anio', 'sector'])['n_ocupados']
    .sum()
    .reset_index()
)

# Pivot para heatmap
pivot = sector_anual.pivot(index='sector', columns='anio', values='n_ocupados')
# Normalizar por fila (% relativo al máximo de cada sector)
pivot_norm = pivot.div(pivot.max(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(15, 6))
im = ax.imshow(pivot_norm.values, aspect='auto', cmap='YlOrRd', vmin=50, vmax=100)

ax.set_xticks(range(len(pivot_norm.columns)))
ax.set_xticklabels(pivot_norm.columns.astype(int), rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot_norm.index)))
ax.set_yticklabels(pivot_norm.index, fontsize=10)
ax.set_title('Ocupados por Sector Económico — Índice relativo al máximo (2005–2026)',
             fontweight='bold', pad=12)

plt.colorbar(im, ax=ax, label='% relativo al año pico de cada sector')
plt.tight_layout()
plt.savefig(FIGS / '04_empleo_sector_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 04_empleo_sector_heatmap.png')

In [ ]:
# ── Salario real promedio por sector (último año disponible) ─────────────────
ultimo_anio = pd_empleo['anio'].max()
sector_sal = (
    pd_empleo[pd_empleo['anio'] == ultimo_anio]
    .groupby('sector')['salario_real_prom']
    .mean()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(sector_sal.index, sector_sal.values,
               color=[COLORES[i % len(COLORES)] for i in range(len(sector_sal))],
               alpha=0.85, height=0.65)

for bar, val in zip(bars, sector_sal.values):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Salario real promedio (pesos base T4-2018)')
ax.set_title(f'Salario Real por Sector Económico — {ultimo_anio}',
             fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig(FIGS / '05_salario_por_sector.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 05_salario_por_sector.png')

---
## 6 · Tasa de desocupación — ciclos económicos

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(pd_kpi['anio'], pd_kpi['tasa_desocupacion_pct'],
        color='#1e3a5f', linewidth=2.5, marker='o', markersize=5, zorder=3)
ax.fill_between(pd_kpi['anio'], pd_kpi['tasa_desocupacion_pct'],
                alpha=0.12, color='#1e3a5f')

# Sombrear recesiones
for inicio, fin, label in [
    (2008.5, 2010, 'Crisis global'),
    (2019.8, 2021, 'COVID-19'),
]:
    ax.axvspan(inicio, fin, alpha=0.12, color='red')
    ax.text((inicio + fin) / 2, pd_kpi['tasa_desocupacion_pct'].max() * 0.97,
            label, ha='center', fontsize=9, color='darkred', style='italic')

# Anotar máximo y mínimo
idx_max = pd_kpi['tasa_desocupacion_pct'].idxmax()
idx_min = pd_kpi['tasa_desocupacion_pct'].idxmin()
for idx, va in [(idx_max, 'bottom'), (idx_min, 'top')]:
    row = pd_kpi.loc[idx]
    ax.annotate(f"{row['tasa_desocupacion_pct']:.1f}%",
                xy=(row['anio'], row['tasa_desocupacion_pct']),
                xytext=(0, 8 if va == 'bottom' else -12),
                textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold',
                color='#1e3a5f')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_xlabel('Año')
ax.set_ylabel('Tasa de desocupación (%)')
ax.set_title('Tasa de Desocupación en México 2005–2026',
             fontweight='bold', pad=12)
ax.set_xlim(2004.5, 2026.5)

plt.tight_layout()
plt.savefig(FIGS / '06_tasa_desocupacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 06_tasa_desocupacion.png')

---
## 7 · IGAE vs salario real — crecimiento económico y bienestar

In [ ]:
# ── Correlación IGAE vs salario real (KPI anual) ─────────────────────────────
kpi_full = pd_kpi.dropna(subset=['igae_anual', 'salario_real_total'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: líneas duales
color_igae = COLORES[3]
color_sal  = COLORES[0]

ax1b = ax1.twinx()
l1, = ax1.plot(kpi_full['anio'], kpi_full['igae_anual'],
               color=color_igae, linewidth=2, label='IGAE (base 2018=100)')
l2, = ax1b.plot(kpi_full['anio'], kpi_full['salario_real_total'],
                color=color_sal, linewidth=2, linestyle='--',
                label='Salario real promedio')
ax1.set_ylabel('IGAE (índice 2018=100)', color=color_igae)
ax1b.set_ylabel('Salario real (pesos)', color=color_sal)
ax1b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.set_title('IGAE vs Salario Real', fontweight='bold')
ax1.legend(handles=[l1, l2], loc='upper left', framealpha=0.9)
ax1.set_xlabel('Año')

# Panel derecho: scatter correlación
sc = ax2.scatter(kpi_full['igae_anual'], kpi_full['salario_real_total'],
                 c=kpi_full['anio'], cmap='viridis', s=80, zorder=3)
# Línea de tendencia
z = np.polyfit(kpi_full['igae_anual'], kpi_full['salario_real_total'], 1)
p = np.poly1d(z)
x_line = np.linspace(kpi_full['igae_anual'].min(), kpi_full['igae_anual'].max(), 100)
ax2.plot(x_line, p(x_line), color='red', linewidth=1.5, linestyle='--', alpha=0.7)

# Anotar algunos años
for _, row in kpi_full[kpi_full['anio'].isin([2005,2010,2015,2020,2026])].iterrows():
    ax2.annotate(str(int(row['anio'])),
                 xy=(row['igae_anual'], row['salario_real_total']),
                 xytext=(4, 4), textcoords='offset points', fontsize=8)

corr = kpi_full['igae_anual'].corr(kpi_full['salario_real_total'])
ax2.set_xlabel('IGAE (índice 2018=100)')
ax2.set_ylabel('Salario real promedio (pesos)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.set_title(f'Correlación IGAE–Salario Real\n(r = {corr:.3f})', fontweight='bold')
plt.colorbar(sc, ax=ax2, label='Año')

plt.tight_layout()
plt.savefig(FIGS / '07_igae_vs_salario.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: 07_igae_vs_salario.png  —  correlación r={corr:.3f}')

---
## 8 · Salario real por nivel educativo

In [ ]:
# ── Evolución por nivel escolar ──────────────────────────────────────────────
sal_escolar = (
    pd_salario
    .groupby(['anio', 'nivel_escolar'])['salario_real_prom']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))

orden = ['Sin instr./Primaria', 'Secundaria/Prepa', 'Técnico/Licenciatura', 'Posgrado']
for i, nivel in enumerate(orden):
    datos = sal_escolar[sal_escolar['nivel_escolar'] == nivel]
    ax.plot(datos['anio'], datos['salario_real_prom'],
            color=COLORES[i], linewidth=2, marker='o', markersize=4, label=nivel)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Año')
ax.set_ylabel('Salario real promedio (pesos base T4-2018)')
ax.set_title('Salario Real por Nivel Educativo 2005–2026', fontweight='bold', pad=12)
ax.legend(framealpha=0.9)
ax.set_xlim(2005, 2026)

plt.tight_layout()
plt.savefig(FIGS / '08_salario_por_escolaridad.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 08_salario_por_escolaridad.png')

---
## 9 · Resumen estadístico y hallazgos

### Hallazgo 1 — El salario real creció, pero de forma desigual e interrumpida

El salario real promedio en México mostró una tendencia positiva entre 2005 y 2026,
con una ganancia acumulada calculada en el pipeline. Sin embargo, este crecimiento
**no fue lineal**: la crisis financiera global de 2008–2009 y la pandemia de COVID-19 en 2020
provocaron caídas significativas del poder adquisitivo, seguidas de recuperaciones.
El período 2021–2023, impulsado por los aumentos al Salario Mínimo General, concentra
la mayor parte de la recuperación reciente.

**Limitación:** El salario real se deflacta con el INPC nacional. Las canastas de consumo
varían por región y nivel socioeconómico, por lo que el impacto real puede diferir
entre grupos de ingreso.

### Hallazgo 2 — La brecha salarial de género persiste, aunque con señales de reducción

A lo largo del período analizado, los hombres percibieron consistentemente salarios
reales más altos que las mujeres. La brecha porcentual muestra una tendencia
descendente desde 2005, especialmente a partir de 2018, sugiriendo avances
en equidad salarial. No obstante, la brecha **no ha cerrado**: en el último año disponible
los hombres siguen ganando más en promedio.

**Limitación:** El análisis mide diferencia de salarios promedios, no brecha ajustada
por ocupación, sector o jornada. Una parte de la brecha puede explicarse por
segregación ocupacional más que por discriminación salarial directa.

### Limitaciones generales del análisis

- **Fuente única:** Todo el análisis proviene de la ENOE (INEGI), que cubre únicamente
  población ocupada en México. No incluye economía informal no captada ni trabajadores
  migrantes.
- **IGAE con rezago:** Los datos del IGAE de los trimestres más recientes pueden estar
  incompletos por el calendario de publicación del INEGI (rezago de 2–3 meses).
- **Deflactor nacional:** Se usa el INPC con base T4-2018 para toda la serie.
  La inflación diferenciada por región o decil de ingreso no se modela.
- **Granularidad trimestral:** La ENOE es una encuesta por hogar con rotación de panel;
  los valores trimestrales son estimaciones con margen de error estadístico.

In [ ]:
# ── Hallazgos clave ──────────────────────────────────────────────────────────
sal_2005 = pd_kpi.loc[pd_kpi['anio'] == 2005, 'salario_real_total'].values[0]
sal_ult  = pd_kpi.loc[pd_kpi['anio'] == pd_kpi['anio'].max(), 'salario_real_total'].values[0]
cambio   = (sal_ult - sal_2005) / sal_2005 * 100

brecha_2005 = pd_brecha[pd_brecha['anio'] == 2005]['brecha_pct'].mean()
brecha_ult  = pd_brecha[pd_brecha['anio'] == pd_brecha['anio'].max()]['brecha_pct'].mean()

desocu_max  = pd_kpi['tasa_desocupacion_pct'].max()
desocu_anio = pd_kpi.loc[pd_kpi['tasa_desocupacion_pct'].idxmax(), 'anio']

print('=' * 60)
print('HALLAZGOS PRINCIPALES — ANÁLISIS GOLD')
print('=' * 60)
print()
print('SALARIO REAL:')
print(f'  2005        : ${sal_2005:,.0f} pesos constantes')
print(f'  {pd_kpi["anio"].max():.0f}        : ${sal_ult:,.0f} pesos constantes')
print(f'  Cambio total: {cambio:+.1f}%')
print()
print('BRECHA SALARIAL:')
print(f'  2005        : {brecha_2005:.1f}% (hombres ganan más)')
print(f'  {pd_brecha["anio"].max():.0f}        : {brecha_ult:.1f}%')
print(f'  Tendencia   : {"mejora" if brecha_ult < brecha_2005 else "empeora"}')
print()
print('DESOCUPACIÓN:')
print(f'  Máximo      : {desocu_max:.1f}% en {desocu_anio:.0f}')
print(f'  Actual      : {pd_kpi["tasa_desocupacion_pct"].iloc[-1]:.1f}%')
print()
print('CORRELACIÓN IGAE–SALARIO REAL:')
print(f'  r = {corr:.3f} ({"alta" if abs(corr) > 0.7 else "moderada"} correlación positiva)')
print()
print(f'Figuras guardadas en: {FIGS}')
print('=' * 60)

spark.stop()